LoRA Fine-Tuning
This notebook fine-tunes a small open LLM using pure LoRA (no QLoRA / no 4-bit quantization) so you can see the standard LoRA workflow clearly.

What you'll see:

GPU check (T4 = 16 GB VRAM)
Load a base model in FP16
Inspect how the base model answers prompts (before fine-tuning)
Prepare a small instruction dataset
Attach LoRA adapters and train
Inspect how the fine-tuned model answers the same prompts (after)
Save adapters
Base model: TinyLlama/TinyLlama-1.1B-Chat-v1.0 — ~1.1B params, ~2.2 GB in FP16.

In [ ]:
!pip install -q \
"transformers==5.0.0" \
"peft==0.18.1" \
"accelerate==1.13.0" \
"datasets==4.8.4" \
"trl==1.1.0" \
"sentencepiece==0.2.1" \
"protobuf==5.29.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.2 MB/s eta 0:00:00


In [ ]:
import sys
import torch

print(torch.__version__)
print(sys.version)

2.11.0+cu128
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
print(torch.cuda.is_available()                 )

True


load base model in FP16

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False
model.config.pretraining_tp = 1

n_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {n_params}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Number of parameters: 1100048384


**Define a prompt and test the base model**

In [ ]:
def build_prompt(schema: str, question: str) -> str:
  system = (
      "You are a SQL assistant. Given a table schema and a question,"
      "reply with ONLY the SQLquery, nothing else."
  )
  user =f"Schema:\n{schema}\n\nQuestion:{question}"
  messages = [
      {"role": "system", "content": system},
      {"role": "user", "content": user}
  ]
  return tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)

In [ ]:
#testing build_prompt function

test_prompt_1 = build_prompt(
    schema ="CREATE TABLE employee (id INT, name TEXT, department TEXT, salary INT);",
    question ="List the names of employees in the Engineering department earning more than 100000."
)
test_prompt_1

'<|system|>\nYou are a SQL assistant. Given a table schema and a question,reply with ONLY the SQLquery, nothing else.</s>\n<|user|>\nSchema:\nCREATE TABLE employee (id INT, name TEXT, department TEXT, salary INT);\n\nQuestion:List the names of employees in the Engineering department earning more than 100000.</s>\n<|assistant|>\n'

In [ ]:
@torch.no_grad()
def generate(prompt: str,max_new_tokens: int = 120) -> str:
  inputs = tokenizer(prompt,return_tensors="pt").to(model.device)

  output_ids = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample = False,
      pad_token_id = tokenizer.pad_token_id,
      eos_token_id = tokenizer.eos_token_id,
  )

  #slice only newly generated tokens
  input_length = inputs["input_ids"].shape[1]
  new_tokens = output_ids[0,input_length:]

  return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
output = generate(test_prompt_1)
print(output)

To answer this question, you can use the following SQL query:

```
SELECT name
FROM employee
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.


In [ ]:
# Three probe prompts we will reuse AFTER fine-tuning for direct comparison.
PROBES = [
    {
        "schema":  "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question":"List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema":  "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question":"What is the total order amount per customer in 2024?",
    },
    {
        "schema":  "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question":"Show the top 5 highest rated horror movies released after 2015.",
    },
]

print("="*70)
print("BASE MODEL (before fine-tuning)")
print("="*70)
base_outputs = []
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    base_outputs.append(ans)
    print(f"\n--- Probe {i} ---")
    print("Q:", p["question"])
    print("A:", ans)


BASE MODEL (before fine-tuning)

--- Probe 1 ---
Q: List the names of employees in the Engineering department earning more than 100000.
A: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE salary > 100000
AND department = 'Engineering';
```

This query will return a list of employees in the Engineering department who have a salary greater than 100,000.

--- Probe 2 ---
Q: What is the total order amount per customer in 2024?
A: To answer this question, you can use the following SQL query:

```
SELECT customer_id, SUM(amount) AS total_order_amount
FROM orders
GROUP BY customer_id
ORDER BY customer_id;
```

This query will group the orders by customer_id and sum the amount for each customer. The `GROUP BY` clause will group the results by customer_id, and the `ORDER BY` clause will order the results by customer_id. The `SUM` function will calculate the total order amount for each customer.

The output

--- Probe 3 ---
Q: Show the top 5 hig

In [ ]:
from datasets import load_dataset

raw = load_dataset("b-mc2/sql-create-context",split="train")
print("dataset size", len(raw))
print("row", raw[0])

#keep it small for a fast and visible demp on t4
raw = raw.shuffle(seed=42).select(range(3000))
split = raw.train_test_split(test_size=0.05,seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("train", len(train_ds), "Eval:", len(eval_ds))

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

dataset size 78577
row {'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}
train 2850 Eval: 150


In [ ]:
#understand dataset
train_ds[0].keys()

dict_keys(['answer', 'question', 'context'])

In [ ]:
print("Schema:", train_ds[0]["context"])
print("="*70)
print("Question:", train_ds[0]["question"])
print("="*70)
print("Answer:", train_ds[0]["answer"])
print("="*70)

Schema: CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)
Question: display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.
Answer: SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")


In [ ]:
def format_example(row):
  system = (
      "You are a SQL assistant. Given a table schema and a question,"
      "reply with ONLY the SQLquery, nothing else."
  )
  user =f"Schema:\n{row['context']}\n\nQuestion:{row['question']}"
  assistant = row["answer"]
  messages = [
      {"role": "system", "content": system},
      {"role": "user", "content": user},
      {"role": "assistant", "content": assistant},
  ]
  text = tokenizer.apply_chat_template(messages,tokenize=False)
  return {"text":text}

train_ds =train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(format_example, remove_columns=eval_ds.column_names)

print("\n--- Formatted training example ---\n")
print(train_ds[0]["text"][:800])
train_ds[0].keys()

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


--- Formatted training example ---

<|system|>
You are a SQL assistant. Given a table schema and a question,reply with ONLY the SQLquery, nothing else.</s>
<|user|>
Schema:
CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)

Question:display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.</s>
<|assistant|>
SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")</s>



dict_keys(['text'])

##
**Attach LoRA Adaptor**

lora insert low rank trianable matrices into specfuc linear lyers (attention projections) while freezing the base weights . For Llama family models the commom target module are q_proj,k_proj,v_proj

In [ ]:
from peft import LoraConfig, get_peft_model

# Enable gradient checkpointing to save VRAM during backward pass
model.gradient_checkpointing_enable()
model.enable_input_require_grads()     # needed because base params are frozen

lora_config = LoraConfig(
    r=16,                                # rank
    lora_alpha=32,                       # scaling factor (alpha / r = 2.0)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# You should see something like: trainable: ~4M / all: ~1.1B  (< 0.5%)

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079



We use SFTTrainer from TRL.

Settings chosen for T4 (16 GB):

per_device_train_batch_size=2, gradient_accumulation_steps=8 → effective batch 16
max_seq_length=512
fp16=True (T4 supports FP16, not BF16)
1 epoch over 3000 examples ≈ ~5–8 minutes


SFTConfig Parameters Explained
output_dir=OUTPUT_DIR
Folder where the fine-tuned model, checkpoints, and logs will be saved.

num_train_epochs=1
Number of times the model sees the full training dataset.

per_device_train_batch_size=2
Number of training samples processed at once on each device.

per_device_eval_batch_size=2
Number of evaluation samples processed at once on each device.

gradient_accumulation_steps=8
Accumulates gradients for 8 steps before updating weights, which gives an effective larger batch size.

gradient_checkpointing=True
Saves GPU memory by recomputing some activations during backpropagation.

learning_rate=2e-4
Controls how big each weight update is during training.

lr_scheduler_type="cosine"
Gradually changes the learning rate using a cosine decay schedule.

warmup_steps=10
Slowly increases the learning rate for the first 10 steps to make training stable.

optim="adamw_torch"
Optimizer used to update model weights, here it is AdamW from PyTorch.

fp16=True
Uses 16-bit floating point precision to reduce memory usage and speed up training.

bf16=False
Disables bfloat16 precision.

logging_steps=10
Logs training details every 10 steps.

eval_strategy="steps"
Runs evaluation during training based on step intervals.

eval_steps=50
Evaluates the model every 50 steps.

save_strategy="epoch"
Saves a checkpoint after each epoch.

report_to="none"
Disables reporting to tools like WandB or TensorBoard.

In [ ]:
from trl import SFTTrainer, SFTConfig
OUTPUT_DIR = "./tinyllama-sql-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",
)

# Pre-tokenize so we don't depend on SFTTrainer's text-field handling
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    return out

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map (tokenize, batched=True, remove_columns=["text"])

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
)

trainer.train()

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
50,0.628503,0.616337
100,0.589141,0.573099
150,0.559597,0.562744


TrainOutput(global_step=179, training_loss=0.7060374600927257, metrics={'train_runtime': 458.3336, 'train_samples_per_second': 6.218, 'train_steps_per_second': 0.391, 'total_flos': 2484042635427840.0, 'train_loss': 0.7060374600927257})

In [ ]:
# Peak VRAM used during training — should stay under T4's 15.8 GB
import torch
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

Peak GPU memory allocated: 2.39 GB


##**Compare AFTER Fine Tuning**

In [ ]:
# Re-enable cache for fast inference
model.config.use_cache = True
model.eval()

print("="*70)
print("FINE-TUNED MODEL (after LoRA)")
print("="*70)
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    print(f"\n--- Probe {i} ---")
    print("Q:     ", p["question"])
    print("BEFORE:", base_outputs[i-1])
    print("="*70)
    print("AFTER :", ans)
    print("="*70)


FINE-TUNED MODEL (after LoRA)

--- Probe 1 ---
Q:      List the names of employees in the Engineering department earning more than 100000.
BEFORE: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE salary > 100000
AND department = 'Engineering';
```

This query will return a list of employees in the Engineering department who have a salary greater than 100,000.
AFTER : SELECT name FROM employees WHERE department = "Engineering" AND salary > 100000

--- Probe 2 ---
Q:      What is the total order amount per customer in 2024?
BEFORE: To answer this question, you can use the following SQL query:

```
SELECT customer_id, SUM(amount) AS total_order_amount
FROM orders
GROUP BY customer_id
ORDER BY customer_id;
```

This query will group the orders by customer_id and sum the amount for each customer. The `GROUP BY` clause will group the results by customer_id, and the `ORDER BY` clause will order the results by customer_id. The `SUM` function w

save the adapters

In [ ]:
ADAPTER_DIR = "./tinyllama-sql-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size on disk: {total/1024**2:.2f} MB")

Adapter size on disk: 20.67 MB


Loading the adapter later (for reference)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, "./tinyllama-sql-lora-adapter")
tokenizer = AutoTokenizer.from_pretrained("./tinyllama-sql-lora-adapter")